**Implementación de modelos supervisados (clasificación/regresión) con Scikit-learn.**

### Importe de librerías necesarias


In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree

from sklearn.preprocessing import FunctionTransformer


In [21]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocesamiento.data_preprocessing import CorrelationFilter
from preprocesamiento.data_preprocessing import FeatureEngineeringRegression
from preprocesamiento.data_preprocessing import tratar_duplicados
from preprocesamiento.data_preprocessing import DataFrameConverter
from preprocesamiento.data_preprocessing import detectar_inconsistencias
from preprocesamiento.data_preprocessing import corregir_valores_negativos
from preprocesamiento.pipelines import build_preprocessor

### Carga de Datos

In [ ]:
url = "https://raw.githubusercontent.com/ramirezluna-david/proyecto_modelado_grp2/rama_david/data/dataset_clientes.csv"
data = pd.read_csv(url)

### Preprocesamiento de datos

#### Detección de inconsistencias

In [8]:
# Deteccion de inconsistencias: negativos o ceros en columnas que deberian ser positivas
conteo_negativos, conteo_ceros = detectar_inconsistencias(data)

Negativos detectados: {'ingreso_mensual': 11, 'gasto_mensual': 68, 'deuda_total': 143}
Ceros detectados: ninguno


In [9]:
# Tratamiento: corrige valores negativos convirtiendolos a valor absoluto
data = corregir_valores_negativos(data)

Valores negativos corregidos a valores absolutos en las columnas relevantes.


#### Elimina duplicados y separa objetivo/features para mantener alineacion

In [10]:
data_sin_dup = tratar_duplicados(data, drop=True).reset_index(drop=True)
var_dep = data_sin_dup["deuda_total"].reset_index(drop=True)
var_indep = data_sin_dup.drop(columns=["deuda_total"])

### Construcción de arrays para procesamiento

In [11]:
numerical_features = ["porcentaje_gasto", "gasto_mensual", "score_crediticio", "ingreso_mensual", "edad", "antiguedad_meses", "frecuencia_compra", "ultima_compra_dias", "num_productos", "hora_registro"] # Define listado de variables numéricas
categorical_nominales = ["abandono", "tiene_tarjeta_credito", "genero", "region", "estado_civil", "canal_registro", "dia_semana_registro"] # Define listado de variables categóricas nominales
categorical_ordinales = ["tipo_plan", "uso_app"] # Define listado de variables categóricas ordinales
date_time_features = ["fecha_registro"] # Define listado de variables de fecha y hora
orden_tipo_plan = ["Basico", "Estandar", "Premium"] # Define orden para variable ordinal tipo_plan
orden_uso_app = ["Bajo", "Medio", "Alto"] # Define orden para variable ordinal uso_app

### Integración de pipelines de transformación


In [12]:
pipeline_numerical_features, pipeline_nominales, pipeline_ordinales, preprocesador = build_preprocessor(
    numerical_features=numerical_features,
    categorical_nominales=categorical_nominales,
    categorical_ordinales=categorical_ordinales,
    orden_tipo_plan=orden_tipo_plan,
    orden_uso_app=orden_uso_app,
 )

# 1. Modelo de regresión lineal (LinealRegression)

## Pipeline para Regresión Lineal

In [14]:
# Define pipeline de limpieza de datos que incluye eliminación de duplicados, ingeniería de características y preprocesamiento específico para cada tipo de variable
pipeline_limpieza = Pipeline(
    steps=[
        ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": True})), # Elimina duplicados del dataset
        ("feature_engineering", FeatureEngineeringRegression()), # Agrega nuevas características
        ("preprocesamiento", preprocesador),
        ("conversion", DataFrameConverter(preprocesador)) # Convierte la salida a DataFrame con nombres
    ]
)

# Define pipeline para regresión lineal
pipeline_modelo_lr = Pipeline(
    steps = [
        ("colinealidad", CorrelationFilter(threshold = 0.9)), # Elimina variables altamente correlacionadas
        ("modelo", LinearRegression()) # Agrega modelo de regresión al pipeline
    ]
)

In [15]:
# Aplica pipeline de limpieza al dataset y obtiene el resultado transformado
data_transformada = pipeline_limpieza.fit_transform(var_indep)
data_transformada.columns = data_transformada.columns.str.replace("num_limpios__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_nom__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_ord__", "")
data_transformada["deuda_total"] = var_dep.to_numpy()
data_transformada.head()

,porcentaje_gasto,gasto_mensual,score_crediticio,ingreso_mensual,edad,antiguedad_meses,frecuencia_compra,ultima_compra_dias,num_productos,hora_registro,...,dia_semana_registro_Domingo,dia_semana_registro_Jueves,dia_semana_registro_Lunes,dia_semana_registro_Martes,dia_semana_registro_Miercoles,dia_semana_registro_Sabado,dia_semana_registro_Viernes,tipo_plan,uso_app,deuda_total
0,8.917517e-02,0.849571,-1.481935,0.219473,0.979150,1.193186,-0.724598,1.653219,0.002765,1.535116,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2.448145e+06
1,-1.246435e+00,-0.584841,-0.260058,1.323554,0.139166,0.140295,-0.358339,1.186639,0.711810,-0.205888,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,1.0,1.620569e+06
2,4.360560e-16,-0.086265,1.738263,0.000000,-0.028831,-1.731512,0.374180,0.472486,0.711810,-0.786223,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,5.395040e+06
3,1.613845e+00,0.119750,-1.611474,-0.949411,0.307163,-1.643771,-1.273987,-0.165490,-0.706279,0.664614,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.999350e+06
4,4.649992e-01,0.623111,-1.351398,-0.220475,-0.980813,-1.351302,-0.907728,0.958110,0.002765,-0.496056,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.637711e+06


In [16]:
target = "deuda_total"

In [17]:
X = data_transformada.drop(columns=[target])
y = data_transformada[target]

## División train/test


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=29)

## Entrenamiento


In [ ]:
pipeline_modelo_lr.fit(X_train, y_train)

In [ ]:
# Predicciones
y_pred = pipeline_modelo_lr.predict(X_test)

# Evaluación
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Métricas del modelo ---")

print(f"{"R2":<6}: {r2:.3f}")
print(f"{"MAE":<6}: {mae:,.0f}")

SyntaxError: unexpected character after line continuation character (4166315165.py, line 10)

In [ ]:
# Variables que fueron eliminadas por presentar colinealidad
pipeline_modelo_lr.named_steps["colinealidad"].columns_to_drop_

In [ ]:
# Variables con las cuales fue entrenado el modelo
pipeline_limpieza.named_steps["conversion"].feature_names_

# Modelo DecisionTreeRegressor

## Pipeline para DecisionTreeRegressor

In [ ]:
# Define pipeline de limpieza de datos que incluye eliminación de duplicados, ingeniería de características y preprocesamiento específico para cada tipo de variable
pipeline_limpieza = Pipeline(
    steps=[
        ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": True})), # Elimina duplicados del dataset
        ("feature_engineering", FeatureEngineeringRegression()), # Agrega nuevas características
        ("preprocesamiento", preprocesador),
        ("conversion", DataFrameConverter(preprocesador)) # Convierte la salida a DataFrame con nombres
    ]
)

# Define pipeline para DecisionTreeRegressor
pipeline_modelo_dtr = Pipeline(
    steps = [
        ("colinealidad", CorrelationFilter(threshold = 0.9)), # Elimina variables altamente correlacionadas
        ("modelo", DecisionTreeRegressor(max_depth = 7, min_samples_leaf = 15, random_state = 42)) # Agrega modelo de regresión al pipeline
    ]
)

In [ ]:
# Aplica pipeline de limpieza al dataset y obtiene el resultado transformado
data_transformada = pipeline_limpieza.fit_transform(var_indep)
data_transformada.columns = data_transformada.columns.str.replace("num_limpios__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_nom__", "")
data_transformada.columns = data_transformada.columns.str.replace("cat_ord__", "")
data_transformada["deuda_total"] = var_dep.to_numpy()
data_transformada.head()

In [ ]:
target = "deuda_total"

In [ ]:
X = data_transformada.drop(columns=[target])
y = data_transformada[target]

## División train/test


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=29)

## Entrenamiento

In [ ]:
pipeline_modelo_dtr.fit(X_train, y_train)

In [ ]:
# Predicciones
y_pred = pipeline_modelo_dtr.predict(X_test)

# Evaluación
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Métricas del modelo ---")

print(f"{\"R2\":<6}: {r2:.3f}")
print(f"{\"MAE\":<6}: {mae:,.0f}")

In [ ]:
modelo_arbol = DecisionTreeRegressor(max_depth = 7, min_samples_leaf = 15, random_state = 42)
modelo_arbol = modelo_arbol.fit(X_train, y_train)

In [ ]:
plt.figure(figsize=(20, 10))

# dibuja el árbol
plot_tree(modelo_arbol,
          feature_names=X_train.columns,  # Muestra los nombres reales de tus variables
          filled=True,
          rounded=True,
          fontsize=10,                    # Tamaño de la letra
          max_depth=3)                    # muestra solo los primeros 3 niveles

# mostrar el gráfico
plt.title("Árbol de Decisión para Regresión", fontsize=15)
plt.show()